# NB-2: HotPotQA — Multi-Model Benchmark + 8B Variance Seeds

Runs CSAM on HotPotQA 2-hop retrieval at 100 questions across all 4 models (seed 42),
then repeats 8B at seeds 123 and 456 for statistical variance.

**Runs:**
- Seed 42 — all 4 models (8B, Scout-17B, 70B, GPT-OSS-120B)
- Seed 123 — 8B only
- Seed 456 — 8B only

**Output directory:** `results/nb2_hotpotqa/`  
**Time estimate:** ~15 min per model run at 100Q

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set API keys

**Kaggle:** Secrets → `GROQ_API_KEY` (+ optionally `GROQ_API_KEY_2` … `GROQ_API_KEY_5`)  
**Colab:** Left sidebar key icon → same secrets

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')

## Step 3 — Configure

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
N_QUESTIONS   = 100   # questions per run (50 for quick test, 100 for publication)
CHECKPOINT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
# ─────────────────────────────────────────────────────────────────────────────

DATASET = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'hotpotqa_dev.json')
OUT_DIR = os.path.join(REPO_DIR, 'results', 'nb2_hotpotqa')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Dataset:    {DATASET}')
print(f'Dataset OK: {os.path.exists(DATASET)}')
print(f'Output dir: {OUT_DIR}')
print(f'Questions:  {N_QUESTIONS} per run')

if not os.path.exists(DATASET):
    data_dir = os.path.dirname(DATASET)
    print(f'[WARN] Dataset missing. Files in {data_dir}:')
    try: print(os.listdir(data_dir))
    except Exception as e: print(f'  {e}')

## Step 4 — Run 1: All 4 models, seed 42, 100Q
This is the primary multi-model comparison run.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
    '--all',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '42',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 1: All 4 models, seed=42, 100Q')
print(f'CMD: {" ".join(cmd[2:])}\n')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 1 complete')

## Step 5 — Run 2: 8B model, seed 123
Variance seed for statistical confidence reporting.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
    '--provider', 'groq',
    '--model', 'llama-3.1-8b-instant',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '123',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 2: Llama-3.1-8B, seed=123, 100Q')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 2 complete')

## Step 6 — Run 3: 8B model, seed 456

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
    '--provider', 'groq',
    '--model', 'llama-3.1-8b-instant',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '456',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 3: Llama-3.1-8B, seed=456, 100Q')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 3 complete')

## Step 7 — Results summary

In [ ]:
import json, os, glob

files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_hotpotqa_*.json')))

print('=' * 70)
print('HOTPOTQA RESULTS')
print('=' * 70)
print(f'{"File":<55} {"Avg F1":>8} {"Sem Sim":>9} {"EM":>6} {"N":>5}')
print('-' * 70)

for fp in files:
    with open(fp) as f: d = json.load(f)
    name = os.path.basename(fp)[:54]
    f1   = d.get('avg_f1', d.get('micro_f1', 0))
    sem  = d.get('avg_semantic_sim', 0)
    em   = d.get('avg_em', d.get('exact_match', 0))
    n    = d.get('num_questions', 0)
    print(f'{name:<55} {f1:>8.4f} {sem:>9.4f} {em:>6.3f} {n:>5}')

print(f'\nTotal result files: {len(files)}')

## Step 8 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb2_hotpotqa'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')